Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# OCI Vault — Secure Secret Retrieval on AI Data Platform

This notebook demonstrates how to securely retrieve secrets (passwords, API keys, connection strings) from **OCI Vault** instead of hardcoding credentials in notebooks or source code. The same `oci` Python package and authentication pattern used here can be applied to interact with any other OCI service (Object Storage, AI Services, Data Catalog, etc.).

## Why OCI Vault?

Hardcoding credentials in notebooks is a security risk — they can be accidentally committed to git, shared, or exposed in saved outputs. OCI Vault solves this by:

- Storing secrets encrypted at rest and in transit
- Controlling access via IAM policies (who or what can read each secret)
- Providing an audit trail of every secret access
- Allowing secret rotation without changing application code

## How Authentication Works

This notebook uses an **OCI config file** stored on your workspace volume. Each developer uploads their own config file and private key once — the notebook reads it automatically on every run.

```
/Volumes/default/default/test/
    └── oci-credentials/
            ├── config          ← OCI config file (generated by oci setup config)
            └── oci_api_key.pem ← your private key
```

## Prerequisites

Before running this notebook, ensure you have:

1. An **OCI Vault** with at least one secret stored
2. The **OCID of the secret** to retrieve (OCI Console → Vault → Secrets)
3. An OCI config file and private key uploaded to your workspace volume

If you don't have the credentials files yet, continue reading — Step 1 walks you through generating and uploading them.

## Step 1 — One-Time Setup: Prepare and Upload OCI Credentials

Each developer does this once. The notebook will then work automatically on every run.

### 1a — Generate your OCI config and key pair

Open **OCI Cloud Shell** (terminal icon, top right of OCI Console) and run:

```bash
oci setup config
```

Follow the prompts:
- **Config location**: press Enter (default `~/.oci/config`)
- **User OCID**: paste yours — OCI Console → Profile → your username
- **Tenancy OCID**: paste yours — OCI Console → Profile → Tenancy
- **Region**: your region (e.g. `sa-saopaulo-1`)
- **Generate new API key**: `Y`
- **Key directory / name**: press Enter (defaults)
- **Passphrase**: press Enter — **leave empty**

Then download all three files from Cloud Shell (gear icon ⚙ → Download):
```
~/.oci/config
~/.oci/oci_api_key.pem
~/.oci/oci_api_key_public.pem
```

---

### 1b — Edit the config file locally

Open the downloaded `config` file in a text editor and update the `key_file` line to the path where you will upload the key on the workspace volume:

```
key_file=/Volumes/<workspace>/<volume>/oci-credentials/oci_api_key.pem
```

---

### 1c — Register the public key in OCI Console

Use the `oci_api_key_public.pem` file downloaded in step 1a to register your API key:

1. Go to OCI Console → Profile → **API Keys** → **Add API Key**.
2. Select **Paste Public Key**, open `oci_api_key_public.pem` in a text editor, and paste its contents.

---

### 1d — Upload both files to the workspace volume

Using the AIDP file browser, upload to your chosen volume path (e.g. `/Volumes/<workspace>/<volume>/oci-credentials/`):
- `config` (already edited in step 1b)
- `oci_api_key.pem`

## Step 2 — Install the OCI SDK

A `requirements.txt` file is included in this folder (`shared-utils/oci_vault/requirements.txt`) with the `oci` package already listed. Use it to install the library via the cluster **Library** tab:

1. Upload `requirements.txt` to your workspace using the AIDP file browser.
2. Open the compute cluster config, go to the **Library** tab, and click **Install Library**.
3. Select **Workspace**, browse to the uploaded `requirements.txt`, and click **Install**.
4. **Restart the cluster** — the `oci` package will be available on every run.

## Step 3 — Configuration

Set the path to your OCI config file on the workspace volume and the OCID of the secret to retrieve.

In [ ]:
# Path to your OCI config file on the workspace volume
OCI_CONFIG_FILE = "/Volumes/default/default/test/config"

# OCID of the secret to retrieve (OCI Console → Vault → Secrets)
SECRET_OCID = "<SECRET_OCID>"

## Step 4 — Load Config and Retrieve Secret

In [ ]:
import oci
import base64

config = oci.config.from_file(file_location=OCI_CONFIG_FILE)
oci.config.validate_config(config)

client = oci.secrets.SecretsClient(config)
response = client.get_secret_bundle(secret_id=SECRET_OCID)
secret_content = response.data.secret_bundle_content.content
secret_value = base64.b64decode(secret_content).decode("utf-8")

print("Secret retrieved successfully.")

## Step 5 — Validate (Without Exposing the Value)

Never print the raw secret value. Validate it was retrieved correctly by checking its length or a masked preview.

In [ ]:
# Validate without exposing the secret
print(f"Secret retrieved — length: {len(secret_value)} characters")
print(f"Preview: {secret_value[:2]}{'*' * (len(secret_value) - 2)}")

## Best Practices

- **Never print the raw secret value** — use length checks or masked previews to validate
- **Use secrets inline** where possible — fetch and pass in one step to minimize exposure time
- **Rotate secrets in Vault** without touching your code — all consumers automatically get the new value on next fetch
- **One secret per credential** — avoid bundling multiple credentials into one secret; keep them separate for fine-grained access control
- **Use IAM policies** to restrict which resources or users can access each secret — follow the principle of least privilege